<a href="https://colab.research.google.com/github/ablamitovroman/data-analysis-practice/blob/main/pandas_exam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Deine Aufgaben (Löse sie Schritt für Schritt):

1. **Bereinigung von `df_kunden`:**
* Finde und entferne doppelte Zeilen (`drop_duplicates`).


2. **Bereinigung und Typkorrektur von `df_trans`:**
* Entferne doppelte Zeilen.
* Bereinige den Spaltennamen oder die Werte im Umsatz, falls nötig, und wandle `"Umsatz"` in einen sauberen Dezimalzahlentyp (`float`) um (Tipp: Nutze `pd.to_numeric` mit `errors="coerce"`, da dort `NaN` und Texte drin stehen). Ersetze verbleibende `NaN`-Umsätze durch `0.0`.


3. **Tabellen zusammenführen (`pd.merge`):**
* Verbinde die bereinigten Kunden mit den Transaktionen über `Kunden_ID` und `User_Code`. Welches `how`-Methoden-Argument wählst du, wenn du **alle Kunden** behalten willst (auch die, die nichts gekauft haben)?
* Entferne den überflüssigen ID-Spaltendb-Namen nach dem Mergen.


4. **Aggregation (`groupby`):**
* Gruppiere das Ergebnis nach `"Stadt"` oder `"Name"` und berechne den **Gesamtumsatz** (`sum`) sowie die **Anzahl der Käufe** (`count`) pro Person/Stadt.



Schreib deinen Code in Colab, probiere es aus und schick mir dein Ergebnis!

In [74]:
import pandas as pd
import io

# Rohdaten 1: Kundendaten (enthält Duplikate und Text-IDs)
kunden_csv = """Kunden_ID,Name,Stadt
101,Anna,München
102,Bernd,Berlin
101,Anna,München
103,Clara,Hamburg
104,Dirk,Stuttgart
105,Eva,Köln"""

# Rohdaten 2: Transaktionen (enthält Text-Umsätze mit Währungssymbolen, NaN-Werte und falsche IDs)
transaktionen_csv = """User_Code,Produkt,Umsatz
101,Laptop,"1200.50 €"
102,Maus,"25.00 €"
101,Tastatur,NaN
106,Monitor,"300.00 €"
103,Stuhl,"150.75 €"
103,Stuhl,"150.75 €"
"""

df_kunden = (
    pd.read_csv(io.StringIO(kunden_csv))
    .drop_duplicates(subset="Kunden_ID")
)

df_trans = (
    pd.read_csv(io.StringIO(transaktionen_csv))
    .drop_duplicates()
    .fillna(0)
)
df_trans['Umsatz'] = (
    df_trans['Umsatz']
    .astype(str)
    .str.replace("€", "", regex=False)
    .astype(float)
)

df_merge = (
    pd.merge(
    df_kunden,
    df_trans,
    left_on="Kunden_ID",
    right_on="User_Code",
    how="left"
    )
    .drop(columns="User_Code")
)

df_merge["Umsatz"] = df_merge["Umsatz"].fillna(0.0)
df_merge["Produkt"] = df_merge["Produkt"].fillna("keine Daten")

print(df_merge)

df_stadt = (
    df_merge
    .groupby("Stadt", as_index=False)
    .agg(
        Gesamtumsatz = ("Umsatz", "sum"),
        Anzahl_der_Kauefen = ("Umsatz", lambda x: (x!=0).sum())
    )
).sort_values(by="Gesamtumsatz", ascending=False)

df_name = (
    df_merge
    .groupby("Name", as_index=False)
    .agg(
        Gesamtumsatz = ("Umsatz", "sum"),
        Anzahl_der_Kauefen = ("Umsatz", lambda x: (x!=0).sum())
    )
).sort_values(by="Gesamtumsatz", ascending=False)
print(df_stadt)
print(df_name)

df_stadt.to_csv("Umsatz_nach_Stadt.csv", index=False)
df_name.to_csv("Umsatz_nach_Name.csv", index=False)

#print(df_kunden)
# print(df_trans)

# print(df_kunden.dtypes)
# print(df_trans.dtypes)

   Kunden_ID   Name      Stadt      Produkt   Umsatz
0        101   Anna    München       Laptop  1200.50
1        101   Anna    München     Tastatur     0.00
2        102  Bernd     Berlin         Maus    25.00
3        103  Clara    Hamburg        Stuhl   150.75
4        104   Dirk  Stuttgart  keine Daten     0.00
5        105    Eva       Köln  keine Daten     0.00
       Stadt  Gesamtumsatz  Anzahl_der_Kauefen
3    München       1200.50                   1
1    Hamburg        150.75                   1
0     Berlin         25.00                   1
2       Köln          0.00                   0
4  Stuttgart          0.00                   0
    Name  Gesamtumsatz  Anzahl_der_Kauefen
0   Anna       1200.50                   1
2  Clara        150.75                   1
1  Bernd         25.00                   1
3   Dirk          0.00                   0
4    Eva          0.00                   0
